# Notebook 04: The DeepHAM policy objective: solutions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yangycpku/Machine_Learning_Macro_PSU/blob/main/Tutorials/Tutorial1/src/04_DeepHAM_Policy_Solutions.ipynb)

**Course:** Penn State Mini-Course on Deep Learning and Heterogeneous Agent Macroeconomics (Penn State University, September 2026)
**Session:** Lecture 1 tutorial: Deep Learning for Solving Heterogeneous Agents Models (DeepHAM)
**Slides:** [`Lectures/`](https://github.com/yangycpku/Machine_Learning_Macro_PSU/tree/main/Lectures) in the course repository
**Notebook role:** solution (to notebook 03)
**Author:** Yucheng Yang (University of Zurich and Swiss Finance Institute). [Course repository](https://github.com/yangycpku/Machine_Learning_Macro_PSU)

---

Everything except the policy objective is imported from the modules you already ran in
notebooks 01 and 02. What you write here is the part that encodes the *economics*: the
household budget constraint, competitive factor prices, and the equilibrium concept.

This is notebook 03 with the five blanks filled. The class body below is byte-identical to
`KSPolicyTrainer` in `policy.py` — it is extracted from that file when this notebook is
generated, so the two cannot disagree.


In [ ]:
RUN_MODE = "smoke"   # one of: "smoke", "teaching", "production"
SEED_INDEX = 3       # which entry of config["random_seed"] to use

## 1. Set up the code directory

DeepHAM is a package of plain Python modules (`param.py`, `dataset.py`, `value.py`,
`policy.py`, ...) that expect to be imported with `src/` as the working directory, with the
data alongside it in `../data`. The cell below handles both ways of running this notebook.

* **Google Colab** (the default for this course). The first run clones the course repository
  into `/content` (about 20 seconds) and moves into `Tutorials/Tutorial1/src`. Nothing needs
  to be installed: Colab already ships TensorFlow, NumPy, SciPy and matplotlib. A GPU is
  optional (`Runtime -> Change runtime type`); at the `smoke` and `teaching` budgets most of
  the wall clock is the NumPy simulation, so the free CPU runtime is fine.
* **A local clone.** Open the notebook from inside `Tutorials/Tutorial1/src` and the cell
  leaves the working directory alone.


In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yangycpku/Machine_Learning_Macro_PSU.git"
REPO_DIR = "/content/Machine_Learning_Macro_PSU"
SRC_DIR = os.path.join(REPO_DIR, "Tutorials", "Tutorial1", "src")

try:
    import google.colab  # noqa: F401  (importable only on a Colab runtime)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isfile(os.path.join(SRC_DIR, "param.py")):
        print("Cloning the course repository into", REPO_DIR, "...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(SRC_DIR)

# Anywhere else (a local clone) the notebook's own folder is already src/.
if not os.path.isfile("param.py"):
    raise FileNotFoundError(
        f"Expected to be inside Tutorials/Tutorial1/src, but the working directory is "
        f"{os.getcwd()!r}. Open this notebook from inside src/, or os.chdir() there."
    )

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Running on Colab:", IN_COLAB)
print("Working directory:", os.getcwd())


In [ ]:
import json
import time
import datetime

import numpy as np
import tensorflow as tf

import simulation_KS as KS
from param import KSParam
from dataset import KSInitDataSet
from value import ValueTrainer
from simulation_KS import simul_shocks, simul_k
from util import print_elapsedtime
from util import set_random_seed

# The subclass defined below lives in this notebook rather than in policy.py, so it
# needs the base class *and* the module-level constants that policy.py defines.
from policy import PolicyTrainer, DTYPE, NP_DTYPE, EPSILON

## 2. Choose the run mode

Everything expensive in DeepHAM is controlled by a handful of numbers. The cell below maps
`RUN_MODE` onto them, so the notebook can be run end to end in a few minutes during class
and at the published setting afterwards.

| | `smoke` | `teaching` | `production` |
|---|---|---|---|
| policy gradient steps | 100 | 1,500 | 10,000 |
| unroll horizon $T$ | 60 | 150 | 150 |
| simulated paths | 64 | 192 | 384 |
| value-net epochs | 10 | 60 | 200 |
| measured wall clock | 2.2 min | 15.6 min | ~90 min |
| mean capital reached | ~11 | ~32 | ~39 (the KS level) |

Timings were measured on an A100 GPU. On Colab's free CPU runtime a `smoke` run takes about
3.5 minutes; most of the wall clock is the NumPy simulation rather than the networks. The capital row summarises what
each budget buys: `smoke` exercises every code path but is far too short to converge,
`teaching` gets most of the way to the Krusell–Smith level of $K \approx 39$, and
`production` reproduces the published results (the reference run shipped in
`../data/simul_results` took 5,278 s).

The cell also keeps two settings consistent: `valid_size` matches `n_path` (the fixed
validation batch is built from `init_ds.datadict`, while its shocks are simulated with
`valid_size` rows), and `batch_size` never exceeds `valid_size`.

In [ ]:
CONFIG_PATH = "./configs/KS/game_nn_n50_0fm1gm.json"   # 0 fixed moments, 1 learned generalized moment
EXP_NAME = "1gm_solution"

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

seed = config["random_seed"][SEED_INDEX]
set_random_seed(seed)
print(f"Solving {CONFIG_PATH} with seed {seed} (index {SEED_INDEX})")
print(
    f'n_fm = {config["n_fm"]} fixed moment(s), '
    f'n_gm = {config["n_gm"]} generalized moment(s), '
    f'{config["n_agt"]} agents'
)

In [ ]:
# Training budget, dispatched on RUN_MODE (see the run-mode cell above).
if RUN_MODE == "smoke":            # exercises every code path, converges to nothing much
    N_PATH, T_BURN = 64, 300
    V_T, V_COUNT, V_EPOCH = 700, 400, 10
    NUM_STEP, T_UNROLL = 100, 60
    FREQ_VALID, FREQ_UPDATE_V = 50, 50
elif RUN_MODE == "teaching":       # gets most of the way to the KS capital stock
    N_PATH, T_BURN = 192, 2000
    V_T, V_COUNT, V_EPOCH = 1200, 600, 60
    NUM_STEP, T_UNROLL = 1500, 150
    FREQ_VALID, FREQ_UPDATE_V = 250, 500
elif RUN_MODE == "production":     # the setting behind the published results
    N_PATH, T_BURN = 384, 6000
    V_T, V_COUNT, V_EPOCH = 2000, 800, 200
    NUM_STEP, T_UNROLL = 10000, 150
    FREQ_VALID, FREQ_UPDATE_V = 500, 2000
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE!r}")

config["dataset_config"]["n_path"] = N_PATH
config["dataset_config"]["t_burn"] = T_BURN
config["value_config"]["T"] = V_T
config["value_config"]["t_count"] = V_COUNT
config["value_config"]["num_epoch"] = V_EPOCH
config["policy_config"]["num_step"] = NUM_STEP
config["policy_config"]["t_unroll"] = T_UNROLL
config["policy_config"]["freq_valid"] = FREQ_VALID
config["policy_config"]["freq_update_v"] = FREQ_UPDATE_V
# valid_size must match n_path: the validation batch comes from init_ds.datadict
# (n_path rows) while its shocks are simulated with valid_size rows.
config["policy_config"]["valid_size"] = N_PATH
config["policy_config"]["batch_size"] = min(config["policy_config"]["batch_size"], N_PATH)
config["value_config"]["batch_size"] = min(config["value_config"]["batch_size"], N_PATH)

assert config["policy_config"]["valid_size"] == config["dataset_config"]["n_path"]
assert config["policy_config"]["batch_size"] <= config["policy_config"]["valid_size"]
assert config["value_config"]["t_count"] < config["value_config"]["T"] - 1, \
    "t_count must leave at least one time slice inside the value simulation"
assert config["policy_config"]["num_step"] >= config["policy_config"]["freq_valid"], \
    "num_step < freq_valid gives zero training epochs (n_epoch = num_step // freq_valid)"

print(
    f"RUN_MODE={RUN_MODE}: {NUM_STEP} policy steps, unroll {T_UNROLL}, "
    f"{N_PATH} paths, {V_EPOCH} value-net epochs"
)

### Where the results go

In [ ]:
mparam = KSParam(config["n_agt"], config["beta"], config["mats_path"])

# The run mode is part of the directory name, so a quick smoke run can never overwrite a
# long production run -- and neither can overwrite the reference solutions shipped in the
# repository (game_nn_n50_1fm1, game_nn_n50_1gm3).
model_path = "../data/simul_results/KS/game_{}_n{}_{}_{}".format(
    config["dataset_config"]["value_sampling"], config["n_agt"], EXP_NAME, RUN_MODE
)
config["model_path"] = model_path
config["current_time"] = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
os.makedirs(model_path, exist_ok=True)
with open(os.path.join(model_path, "config_beg.json"), "w") as f:
    json.dump(config, f)
print("Results will be written to", model_path)

## 3. Build the initial dataset

`KSInitDataSet` first burns in a panel of `n_path` economies of `n_agt` agents under the
Krusell–Smith benchmark policy, so that the cross-section of wealth starts from its
ergodic distribution. The value networks are then trained on data simulated from an
*initial* policy: either that same benchmark policy (`init_with_bchmk = true`) or a
constant consumption share (`init_with_bchmk = false`, the setting used here).

In [ ]:
start_time = time.monotonic()

init_ds = KSInitDataSet(mparam, config)
value_config = config["value_config"]

if config["init_with_bchmk"]:
    init_policy = init_ds.k_policy_bchmk        # the KS benchmark (b-spline) policy
    policy_type = "pde"
else:
    init_policy = init_ds.c_policy_const_share  # a constant consumption share
    policy_type = "nn_share"

# Supervised targets for the value nets: discounted utility along simulated paths.
train_vds, valid_vds = init_ds.get_valuedataset(init_policy, policy_type, update_init=False)
print_elapsedtime(time.monotonic() - start_time)

## 4. Pre-train the value networks

`num_vnet` independent value networks are fitted to the same targets. They are used as the
terminal bootstrap $\beta^{T}V(s_T)$ that closes the finite unroll in the policy objective;
averaging several of them reduces the variance of that bootstrap.

In [ ]:
vtrainers = []
for i in range(value_config["num_vnet"]):
    config["vnet_idx"] = str(i)
    vtrainers.append(ValueTrainer(config))

for i, vtr in enumerate(vtrainers):
    print(f"--- value net {i} ---")
    vtr.train(train_vds, valid_vds, value_config["num_epoch"], value_config["batch_size"])

## 5. The policy trainer, complete

Three details are worth pausing on.

**The equilibrium concept lives in one line.** `tf.stop_gradient(c_share[:, 1:])` fixes the
other agents' behaviour while agent 0 optimises. Delete it and the objective becomes a
social planner's problem — which is exactly the `socialplanner` branch at the bottom, where
the mean is taken over *all* agents rather than agent 0 alone.

**The clip is doing real work.** `tf.clip_by_value(c_share * wealth, EPSILON, wealth - EPSILON)`
keeps consumption strictly inside $(0, \text{wealth})$, so $\log c$ stays finite and next
period's capital stays positive during the unroll — including early in training, when the
policy network is close to random.

**The horizon is closed, not truncated.** At `t == t_unroll - 1` the loop stops accumulating
utility and adds $\beta^{T} \overline{V}(s_T)$ instead, averaged over the value networks.
That is why `t_unroll = 150` is enough even though the model is infinite-horizon.

In [ ]:
class KSPolicyTrainer(PolicyTrainer):
    def __init__(self, vtrainers, init_ds, policy_path=None):
        super().__init__(vtrainers, init_ds, policy_path)
        if self.config["init_with_bchmk"]:
            init_policy = self.init_ds.k_policy_bchmk
            policy_type = "pde"
        else:
            init_policy = self.init_ds.c_policy_const_share
            policy_type = "nn_share"
        self.policy_ds = self.init_ds.get_policydataset(init_policy, policy_type, update_init=False)

    @tf.function
    def loss(self, input_data):
        k_cross = input_data["k_cross"]
        ashock, ishock = input_data["ashock"], input_data["ishock"]
        util_sum = 0

        for t in range(self.t_unroll):
            k_mean = tf.reduce_mean(k_cross, axis=1, keepdims=True)
            k_mean_tmp = tf.tile(k_mean, [1, self.mparam.n_agt])
            k_mean_tmp = tf.expand_dims(k_mean_tmp, axis=-1)
            i_tmp = ishock[:, :, t:t+1] # n_path*n_agt*1
            a_tmp = tf.tile(ashock[:, t:t+1], [1, self.mparam.n_agt])
            a_tmp = tf.expand_dims(a_tmp, axis=2) # n_path*n_agt*1
            basic_s_tmp = tf.concat([tf.expand_dims(k_cross, axis=-1), k_mean_tmp, a_tmp, i_tmp], axis=-1)
            basic_s_tmp = self.init_ds.normalize_data(basic_s_tmp, key="basic_s", withtf=True)
            full_state_dict = {
                "basic_s": basic_s_tmp,
                "agt_s": self.init_ds.normalize_data(tf.expand_dims(k_cross, axis=-1), key="agt_s", withtf=True)
            }
            if t == self.t_unroll - 1:
                value = 0
                for vtr in self.vtrainers:
                    value += self.init_ds.unnormalize_data(
                        vtr.value_fn(full_state_dict)[..., 0], key="value", withtf=True)
                value /= self.num_vnet
                util_sum += self.discount[t]*value
                continue

            c_share = self.policy_fn(full_state_dict)[..., 0]
            if self.policy_config["opt_type"] == "game":
                # optimizing agent 0 only
                c_share = tf.concat([c_share[:, 0:1], tf.stop_gradient(c_share[:, 1:])], axis=1)
            # labor tax rate - depend on ashock
            tau = tf.where(ashock[:, t:t+1] < 1, self.mparam.tau_b, self.mparam.tau_g)
            # total labor supply - depend on ashock
            emp = tf.where(
                ashock[:, t:t+1] < 1,
                self.mparam.l_bar*self.mparam.er_b,
                self.mparam.l_bar*self.mparam.er_g
            )
            tau, emp = tf.cast(tau, DTYPE), tf.cast(emp, DTYPE)
            R = 1 - self.mparam.delta + ashock[:, t:t+1] * self.mparam.alpha*(k_mean / emp)**(self.mparam.alpha-1)
            wage = ashock[:, t:t+1]*(1-self.mparam.alpha)*(k_mean / emp)**(self.mparam.alpha)
            wealth = R * k_cross + (1-tau)*wage*self.mparam.l_bar*ishock[:, :, t] + \
                self.mparam.mu*wage*(1-ishock[:, :, t])
            csmp = tf.clip_by_value(c_share * wealth, EPSILON, wealth-EPSILON)
            k_cross = wealth - csmp
            util_sum += self.discount[t] * tf.math.log(csmp)

        if self.policy_config["opt_type"] == "socialplanner":
            output_dict = {
                "m_util": -tf.reduce_mean(util_sum), 
                "k_end": tf.reduce_mean(k_cross)
                }
        elif self.policy_config["opt_type"] == "game":
            # optimizing agent 0 only
            output_dict = {
                "m_util": -tf.reduce_mean(util_sum[:, 0]),
                "k_end": tf.reduce_mean(k_cross)
                }
        return output_dict

    def update_policydataset(self, update_init=False):
        self.policy_ds = self.init_ds.get_policydataset(self.current_c_policy, "nn_share", update_init)

    def get_valuedataset(self, update_init=False):
        return self.init_ds.get_valuedataset(self.current_c_policy, "nn_share", update_init)

    def current_c_policy(self, k_cross, ashock, ishock):
        k_mean = np.mean(k_cross, axis=1, keepdims=True)
        k_mean = np.repeat(k_mean, self.mparam.n_agt, axis=1)
        ashock = np.repeat(ashock, self.mparam.n_agt, axis=1)
        basic_s = np.stack([k_cross, k_mean, ashock, ishock], axis=-1)
        basic_s = self.init_ds.normalize_data(basic_s, key="basic_s")
        basic_s = basic_s.astype(NP_DTYPE)
        full_state_dict = {
            "basic_s": basic_s,
            "agt_s": self.init_ds.normalize_data(k_cross[:, :, None], key="agt_s")
        }
        c_share = self.policy_fn(full_state_dict)[..., 0]
        return c_share

    def simul_shocks(self, n_sample, T, mparam, state_init):
        return KS.simul_shocks(n_sample, T, mparam, state_init)

## 6. Train

In [ ]:
policy_config = config["policy_config"]
ptrainer = KSPolicyTrainer(vtrainers, init_ds)
ptrainer.train(policy_config["num_step"], policy_config["batch_size"])

## 7. Save, and check that the run produced something sane

In [ ]:
with open(os.path.join(model_path, "config.json"), "w") as f:
    json.dump(config, f)

for i, vtr in enumerate(vtrainers):
    vtr.save_model(os.path.join(model_path, "value{}.weights.h5".format(i)))
ptrainer.save_model(os.path.join(model_path, "policy.weights.h5"))

elapsed = time.monotonic() - start_time
with open(os.path.join(model_path, "time.txt"), "w") as f:
    f.write(f"{CONFIG_PATH} at RUN_MODE={RUN_MODE} took {elapsed:.2f} seconds.\n")

print_elapsedtime(elapsed)
print("Saved to", model_path)

### Keeping your results (Colab)

On Colab, `model_path` lives on the runtime's own disk and disappears when the runtime is
recycled. Flip the switch below to download a zip of the run to your computer (or copy the
folder to Google Drive after mounting it from the file browser on the left).


In [ ]:
DOWNLOAD_RESULTS = False   # set to True on Colab to download a zip of this run

if DOWNLOAD_RESULTS and IN_COLAB:
    import shutil
    from google.colab import files
    zip_path = shutil.make_archive(model_path, "zip", model_path)
    files.download(zip_path)
    print("Downloading", zip_path)


In [ ]:
# End-to-end check: the saved policy should simulate to a finite, economically plausible
# capital stock. This is a sanity check on the pipeline, not on convergence -- a `smoke`
# run is far too short to be accurate.
for fname in ["policy.weights.h5", "config.json", "stats.json"]:
    assert os.path.exists(os.path.join(model_path, fname)), f"missing {fname} in {model_path}"

_state = init_ds.next_batch(16)
_shocks = simul_shocks(16, 50, mparam, _state)
_sim = simul_k(
    16, 50, mparam, ptrainer.current_c_policy,
    policy_type="nn_share", state_init=_state, shocks=_shocks,
)
_K = _sim["k_cross"].mean()
assert np.isfinite(_K) and 1.0 < _K < 500.0, f"implausible mean capital {_K}"
print(f"Check passed: mean capital over a short simulation = {_K:.2f}")

## Takeaway

Compare the objective here with a value-function-iteration solver: there is no Bellman
operator, no grid, and no interpolation. The policy is improved by differentiating through
a *simulation* of the economy, which is what lets the state include a learned function of
the whole wealth distribution rather than a handful of grid dimensions.